# Lab 12 solution, notebook 1: ask in other words, and add a note

The worked answers to the two challenge tasks of `12a_meet_retrieval.ipynb`. The setup repeats what the tasks need
from the notebook (the seven notes, the collection, `embed`, `search` and `answer`), so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import numpy as np
import chromadb
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

NOTES = {
    "wifi": "The wifi password is on a sticker on the fridge door.",
    "bins": "Bins go out on Tuesday night; the truck comes early on Wednesday.",
    "parking": "Guests park on Hill Street, outside number 12.",
    "boiler": "If the hot water stops, press the red reset button on the boiler.",
    "quiet": "Quiet hours start at 22:00 on weeknights.",
    "neighbours": "Omar and Lina in flat 4 keep a spare key to our flat.",
    "window_keys": "Spare keys for the windows are in the kitchen drawer.",
}
names = list(NOTES)
response = client.embeddings.create(model=EMBEDDING_MODEL, input=list(NOTES.values()))
D = np.array([item.embedding for item in response.data])

chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection("house_notes", configuration={"hnsw": {"space": "cosine"}})
collection.add(ids=names, documents=list(NOTES.values()), embeddings=D.tolist())
print("stored:", collection.count())

stored: 7


The same three functions as the notebook: `embed` returns one vector, `search` returns the nearest notes as
dictionaries, and `answer` packs them and asks the model to answer only from them.

In [3]:
RULES = "Answer only from the notes. If the notes do not say, say you do not know."


def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding


def search(question, k=3):
    got = collection.query(query_embeddings=[embed(question)], n_results=k)
    rows = []
    for note_id, text, distance in zip(got["ids"][0], got["documents"][0], got["distances"][0]):
        rows.append({"id": note_id, "text": text, "cosine": round(1 - distance, 3)})
    return rows


def answer(question, k):
    found = search(question, k)
    packed = "\n".join(f"[{row['id']}] {row['text']}" for row in found)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RULES},
            {"role": "user", "content": f"Notes:\n{packed}\n\nQuestion: {question}"},
        ],
        reasoning_effort=EFFORT or omit,
    )
    return [row["id"] for row in found], response.choices[0].message.content

---

## Task 1: Ask the locked-out question in other words

First the search, with the three nearest notes and their cosines.

In [4]:
LOCKED_OUT = "I'm locked out. Who can let me in?"

for row in search(LOCKED_OUT, k=3):
    print(row["cosine"], row["id"])

0.3 window_keys
0.275 neighbours
0.239 wifi


`window_keys` comes first again (0.3), and `neighbours` second (0.275). The question names no key at all, and the note about window keys still ranks above the note that names the people who can let the guest in.

So the shared words *spare key* are not what put the window note first in Part F. The search ranks by closeness of meaning, and its closest note is not
always the useful one.

Now the answer with only the nearest note:

In [5]:
used, reply = answer(LOCKED_OUT, k=1)
print("notes used:", used)
print(reply)

notes used: ['window_keys']
I don’t know who can let you in. Spare window keys are in the kitchen drawer.


In our run: *I don’t know who can let you in. Spare window keys are in the kitchen drawer.* The same miss as Part F of the notebook: with `k=1` the guest is not helped, and nothing raised an error. With `k=2` the neighbours' note, second in the search above, would be in the prompt too.

---

## Task 2: Add a note to the collection

We embed the new note with `embed` and add it with the same three fields as before, each in a list of one.

In [6]:
BOTTLES = "Glass bottles go in the green box by the back door."

collection.add(ids=["recycling"], documents=[BOTTLES], embeddings=[embed(BOTTLES)])
print("stored:", collection.count())

stored: 8


In [7]:
WINE = "Where does an empty wine bottle go?"

for row in search(WINE, k=3):
    print(row["cosine"], row["id"])

used, reply = answer(WINE, k=1)
print("notes used:", used)
print(reply)

0.555 recycling
0.213 bins
0.203 window_keys


notes used: ['recycling']
In the green box by the back door.


`stored: 8`. The new note comes first for the wine-bottle question (0.555, far above `bins` at 0.213), sharing little more than the word *bottle* with it, and the reply is *In the green box by the back door.*

Adding a note is one `add`: nothing else in the collection changes, and the next search ranks it with the others.
The note's vector must come from the same embedding model as the rest, or its cosines mean nothing.